# LOGISTICS-04 — Etapa 2.25-A: Investigação de `aoi_id` e `courier_id`

**Objetivo de negócio:** verificar se informações de localização detalhada (AOI) e do entregador aumentam a capacidade de priorizar coletas com risco de atraso **no instante do aceite**.

**Origem metodológica:** `03_ml_shanghai.ipynb`, etapas 2.5–2.9, 2.17, 2.23 e 2.24. Este é um **notebook novo e independente**: não altera o histórico. Executar as células de cima para baixo, uma por vez.

**Experimentos:** (A) 10 features originais; (B) 10 + `aoi_id`; (C) 10 + `aoi_id` + `courier_id`. Executar os mesmos Random Forest e XGBoost de 2.23 nos três cenários (6 ajustes). Métrica principal: Average Precision. Operação: Precision@K e Recall@K para 1, 2, 5, 10, 15, 20 e 30%.

**Proteção metodológica:** teste não será carregado, transformadores ajustados apenas no treinamento; avaliação na validação original. Para evitar matrizes densas potencialmente enormes com identificadores de alta cardinalidade, os três cenários usam **OneHotEncoder esparso**. Logo, o baseline é **reexecutado** sob a mesma representação para comparação interna; os números históricos de 2.24, produzidos com matriz densa, são referência, não comparação exatamente controlada.

**Atenção:** IDs são categorias, não quantidades contínuas. Um ganho em validação não prova causalidade nem que a região pesa mais. A seleção de features na validação também implica seleção de modelos: manter o teste reservado para avaliação final.

## 1 — Preparação do ambiente

Verificar versões e localizar a Silver, sem modificar dependências ou arquivos. Execute esta célula primeiro.

In [3]:
from pathlib import Path
import sys, time, gc
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent
silver = raiz / "data" / "silver" / "pickup_shanghai_silver.parquet"
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "sklearn:", sklearn.__version__, "xgboost:", xgboost.__version__)
print("Silver:", silver)
assert silver.is_file(), "Silver não encontrada: confira a pasta de execução."
print("Arquivo localizado. Nenhum modelo treinado.")

Python: 3.13.16
pandas: 2.3.3 sklearn: 1.9.1 xgboost: 3.4.1
Silver: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\data\silver\pickup_shanghai_silver.parquet
Arquivo localizado. Nenhum modelo treinado.


### Entendendo o Python — ambiente e dependências

**Bibliotecas e responsabilidades**

- `pathlib.Path`: representa caminhos de arquivos como objetos; `/` concatena partes do caminho e `Path.cwd()` identifica a pasta de execução.
- `sys.version`: permite registrar a versão do interpretador, importante para reprodutibilidade.
- `pandas` e `numpy`: trabalham com tabelas e vetores numéricos; `sklearn` e `xgboost` fornecem os modelos e transformações.
- `train_test_split`: separa registros; `ColumnTransformer` aplica transformações distintas a colunas diferentes; `Pipeline` encadeia as etapas.
- `time.perf_counter()`: cronômetro adequado para medir duração de operações; `gc` auxilia na liberação de objetos sem referências.
- `display()`: exibe DataFrames formatados no Jupyter.

**Fluxo do código:** primeiro importa as bibliotecas; depois identifica a raiz do projeto quando o notebook está na raiz ou em `notebooks/`; por fim monta o caminho da Silver e confirma sua existência com `assert`. Se o arquivo não existir, a execução para com uma mensagem explícita. **Nenhum dado é alterado e nenhum modelo é treinado.**

### Etapa 1 — Resultados, interpretação e conclusão

#### Resultados obtidos

A preparação do ambiente foi executada com sucesso, confirmando:

- **Python:** 3.13.16
- **Pandas:** 2.3.3
- **Scikit-learn:** 1.9.1
- **XGBoost:** 3.4.1
- **Dataset Silver:** `data/silver/pickup_shanghai_silver.parquet`, localizado com sucesso.

Nenhum modelo foi treinado e nenhuma transformação dos dados foi realizada nesta etapa.

#### Interpretação

A verificação confirmou a disponibilidade das bibliotecas necessárias para os experimentos de classificação e a existência do arquivo Silver.

Essa validação inicial reduz o risco de interrupções causadas por dependências ausentes ou caminhos incorretos.

A existência do arquivo não comprova, por si só, a integridade de seu conteúdo, que será verificada nas etapas seguintes.

#### Conclusão

**Etapa 1 concluída com sucesso.**

O ambiente está preparado para prosseguir com a leitura e a inspeção do dataset Silver, preservando os experimentos anteriores e o conjunto de teste reservado para a avaliação final.

## 2 — Leitura, elegibilidade e auditoria dos identificadores

Reproduzir a população elegível da etapa 2.7 e inspecionar nulos, cardinalidade e frequência de `region_id`, `aoi_id` e `courier_id`. Não excluir AOIs ou entregadores raros automaticamente.

In [4]:
baseline = ["region_id", "aoi_type", "lng", "lat", "janela_min", "aceite_ate_inicio_janela_min", "aceite_ate_fim_janela_min", "hora_aceite", "hora_inicio_janela", "hora_fim_janela"]
colunas_silver = ["order_id", "accept_time", "time_window_start", "time_window_end", "coleta_apos_limite", "region_id", "aoi_type", "lng", "lat", "janela_min", "aceite_ate_inicio_janela_min", "aceite_ate_fim_janela_min", "aoi_id", "courier_id"]
df = pd.read_parquet(silver, columns=colunas_silver)
# Reproduz a máscara usada na etapa 2.7; validar também campos essenciais.
assert df[["accept_time", "time_window_end", "coleta_apos_limite"]].notna().all().all(), "Revisar campos essenciais ausentes antes de prosseguir."
df = df.loc[df["accept_time"] <= df["time_window_end"]].copy()
df["hora_aceite"] = df["accept_time"].dt.hour
df["hora_inicio_janela"] = df["time_window_start"].dt.hour
df["hora_fim_janela"] = df["time_window_end"].dt.hour
assert len(df) == 1_424_195, "População diferente da etapa 2.8: investigar antes de continuar."
assert not df[baseline].isna().any().any(), "Features do baseline com nulos: investigar."
assert set(df["coleta_apos_limite"].dropna().unique()) == {0, 1}
print("Elegíveis:", len(df), "| Atrasos:", int(df["coleta_apos_limite"].sum()))
auditoria = pd.DataFrame([{"Variável": c, "Distintos": df[c].nunique(dropna=True), "Nulos": int(df[c].isna().sum()), "Frequência mínima": int(df[c].value_counts().min()) if df[c].notna().any() else 0, "Categorias com menos de 20 registros": int((df[c].value_counts() < 20).sum())} for c in ["region_id", "aoi_id", "courier_id"]])
display(auditoria)
assert not df[["aoi_id", "courier_id"]].isna().any().any(), "IDs ausentes: definir tratamento documentado antes de treinar."
print("Auditoria concluída; nenhum registro descartado por raridade.")

Elegíveis: 1424195 | Atrasos: 36319


,Variável,Distintos,Nulos,Frequência mínima,Categorias com menos de 20 registros
0,region_id,30,0,4167,0
1,aoi_id,5867,0,1,2140
2,courier_id,4493,0,1,2395


Auditoria concluída; nenhum registro descartado por raridade.


### Entendendo o Python — seleção de colunas, filtros e auditoria

1. `baseline` é uma **lista** de nomes das dez features originais. A ordem é mantida para reproduzir a preparação anterior.
2. `pd.read_parquet(..., columns=...)` carrega somente as colunas necessárias da Silver, reduzindo leitura e memória.
3. `df[[...]].notna().all().all()` testa se os campos essenciais estão preenchidos; `assert` interrompe o experimento se a premissa falhar.
4. `df.loc[condição].copy()` aplica a regra de elegibilidade: o aceite não pode ocorrer depois do fim da janela. `.copy()` cria um DataFrame independente.
5. `.dt.hour` extrai a hora de valores `datetime` para reproduzir as features temporais do baseline.
6. `nunique(dropna=True)` conta categorias distintas; `isna().sum()` conta ausências; `value_counts()` mede frequência por identificador. A expressão `(frequências < 20).sum()` conta categorias com poucos exemplos.
7. As verificações de contagem, classes do alvo e ausência de nulos funcionam como **contratos de integridade**: em caso de divergência, investigar antes de prosseguir.

**Conceito:** cardinalidade é a quantidade de valores diferentes de uma variável. Identificadores de AOI e entregador podem gerar muitas colunas após One-Hot Encoding. Nesta etapa apenas investigamos; não removemos categorias raras.

### Etapa 2 — Resultados, interpretação e conclusão

#### Resultados obtidos

A auditoria identificou **1.424.195 registros elegíveis**, dos quais **36.319 correspondem a coletas atrasadas**, representando aproximadamente **2,55%** do total.

A análise das variáveis geográficas e operacionais apresentou:

| Variável | Categorias distintas | Categorias com menos de 20 registros | Valores nulos |
|---|---:|---:|---:|
| `region_id` | 30 | 0 | 0 |
| `aoi_id` | 5.867 | 2.140 | 0 |
| `courier_id` | 4.493 | 2.395 | 0 |

Nenhum registro foi excluído em função da baixa frequência de determinadas categorias.

#### Interpretação

A variável `region_id` apresenta baixa cardinalidade e distribuição relativamente consistente, com pelo menos 4.167 registros por categoria.

As variáveis `aoi_id` e `courier_id` apresentam alta cardinalidade e um número expressivo de categorias com poucas observações.

Esse comportamento exige atenção porque modelos podem aprender associações específicas de categorias pouco representadas, sem que essas associações se generalizem adequadamente para novas coletas.

Entretanto, a alta cardinalidade não significa que essas variáveis sejam irrelevantes. Elas podem fornecer informações operacionais e geográficas adicionais que não são capturadas pela região.

A inclusão dessas variáveis deverá ser avaliada empiricamente, comparando o desempenho preditivo dos modelos em condições equivalentes.

#### Conclusão

**Etapa 2 concluída com sucesso.**

A auditoria confirmou a disponibilidade das variáveis necessárias aos três cenários experimentais.

A decisão metodológica é preservar `region_id` no baseline e investigar a contribuição incremental de `aoi_id` e `courier_id`, sem excluir categorias raras antecipadamente.

O conjunto de teste permanece reservado para a avaliação final.

## 3 — Reconstrução do particionamento histórico

Repetir **exatamente** os dois `train_test_split` da etapa 2.8 (`random_state=42`, estratificação). Não criar um novo particionamento. A variável `idx_teste` será usada **somente para checar separação**, sem selecionar features, alvos ou fazer previsões de teste.

In [5]:
indices = df.index
y = df["coleta_apos_limite"].astype(int)
idx_treino, idx_temporario = train_test_split(indices, test_size=0.30, stratify=y, random_state=42)
idx_validacao, idx_teste = train_test_split(idx_temporario, test_size=0.50, stratify=y.loc[idx_temporario], random_state=42)
assert len(idx_treino) == 996_936 and len(idx_validacao) == 213_629 and len(idx_teste) == 213_630
assert set(idx_treino).isdisjoint(idx_validacao) and set(idx_treino).isdisjoint(idx_teste) and set(idx_validacao).isdisjoint(idx_teste)
treino = df.loc[idx_treino].copy()
validacao = df.loc[idx_validacao].copy()
y_tr = treino["coleta_apos_limite"].astype(int)
y_va = validacao["coleta_apos_limite"].astype(int)
print("Treino:", len(treino), "| Validação:", len(validacao), "| Teste reservado:", len(idx_teste))
print("Taxa de atraso treino:", round(y_tr.mean(), 5), "| validação:", round(y_va.mean(), 5))
del df, y, indices, idx_temporario
gc.collect()

Treino: 996936 | Validação: 213629 | Teste reservado: 213630
Taxa de atraso treino: 0.0255 | validação: 0.0255


80

### Entendendo o Python — estratificação e índices

- `df.index` fornece os identificadores posicionais das linhas do DataFrame carregado; separar índices evita copiar todas as features durante o sorteio.
- `y` é a variável-alvo binária (`coleta_apos_limite`); `.astype(int)` padroniza seu tipo.
- O primeiro `train_test_split(..., test_size=0.30)` reserva 30% dos índices; o segundo divide essa reserva ao meio, formando 70% treino, 15% validação e 15% teste.
- `stratify=y` mantém aproximadamente a proporção de atrasos em cada partição; `random_state=42` fixa a aleatoriedade. **A mesma ordem das linhas e a mesma população são necessárias** para reconstruir o sorteio anterior.
- `set(...).isdisjoint(...)` confirma que os conjuntos não compartilham índices.
- `.loc[idx]` recupera apenas treino e validação. `idx_teste` é mantido exclusivamente para conferência de tamanho e separação, sem acessar suas features ou seu alvo.
- `del` remove referências a objetos intermediários e `gc.collect()` solicita coleta de lixo; isso não garante liberação imediata de toda a memória.

**Conceito:** estratificação preserva a frequência da classe, mas não impõe separação por entregador ou região. Essa é uma escolha de validação compatível com novas coletas na operação conhecida.

### Etapa 3 — Resultados, interpretação e conclusão

#### Resultados obtidos

O particionamento dos 1.424.195 registros elegíveis apresentou:

| Conjunto | Registros | Participação |
|---|---:|---:|
| Treinamento | 996.936 | 70% |
| Validação | 213.629 | 15% |
| Teste | 213.630 | 15% |

A taxa de atraso foi de aproximadamente **2,55%** nos conjuntos de treinamento e validação.

#### Interpretação

A distribuição obtida confirma que a estratificação preservou aproximadamente a proporção da variável-alvo nos conjuntos utilizados para treinamento e validação.

A manutenção dessas proporções é importante porque estamos trabalhando com um problema de classificação desbalanceada, no qual as coletas atrasadas representam uma parcela pequena dos registros.

O particionamento também reproduziu as quantidades dos experimentos anteriores, favorecendo a comparabilidade dos resultados.

Entretanto, a correspondência das quantidades não comprova, isoladamente, que os mesmos registros foram atribuídos aos mesmos conjuntos. Essa equivalência depende da reprodução dos critérios de particionamento.

#### Conclusão

**Etapa 3 concluída com sucesso quanto às quantidades e à estratificação.**

Os conjuntos estão preparados para os experimentos comparativos com diferentes configurações de features.

O conjunto de teste permanece reservado, sem utilização na seleção dos modelos.

## 4 — Configuração dos cenários e do custo computacional

Definir as três seleções de features. `aoi_id` e `courier_id` entram como **categóricas**. Os parâmetros dos dois classificadores são os da etapa 2.23. Para controlar memória, usar One-Hot esparso e executar sequencialmente, liberando cada pipeline após a avaliação.

In [6]:
cenarios = {
    "A — Baseline (10)": baseline,
    "B — Baseline + AOI (11)": baseline + ["aoi_id"],
    "C — Baseline + AOI + Courier (12)": baseline + ["aoi_id", "courier_id"],
}
cat_base = ["region_id", "aoi_type"]
capacidades = [1, 2, 5, 10, 15, 20, 30]

def montar_pipeline(nome, features):
    cat = cat_base + [c for c in ["aoi_id", "courier_id"] if c in features]
    num = [c for c in features if c not in cat]
    pre = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat),
        ("numericas", StandardScaler(), num),
    ], sparse_threshold=1.0)
    if nome == "Random Forest":
        estimador = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20, max_features="sqrt", class_weight=None, n_jobs=4, random_state=42)
    else:
        estimador = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", tree_method="hist", n_jobs=4, random_state=42)
    return Pipeline([("preprocessamento", pre), ("classificador", estimador)])

for nome, fs in cenarios.items():
    assert len(fs) == len(set(fs))
    assert set(fs).issubset(treino.columns)
    print(nome, ":", fs)
print("Total planejado: 3 cenários × 2 algoritmos = 6 treinamentos")

A — Baseline (10) : ['region_id', 'aoi_type', 'lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela']
B — Baseline + AOI (11) : ['region_id', 'aoi_type', 'lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela', 'aoi_id']
C — Baseline + AOI + Courier (12) : ['region_id', 'aoi_type', 'lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela', 'aoi_id', 'courier_id']
Total planejado: 3 cenários × 2 algoritmos = 6 treinamentos


### Entendendo o Python — cenários, codificação e pipeline

- `cenarios` é um **dicionário**: cada chave nomeia um experimento e cada valor contém sua lista de features. `baseline + ["aoi_id"]` cria uma nova lista, sem modificar o baseline original.
- `cat_base` define variáveis categóricas; a **list comprehension** `[...] if c in features` acrescenta AOI e Courier somente quando presentes no cenário. As demais features são numéricas.
- `ColumnTransformer` aplica `OneHotEncoder` às categorias e `StandardScaler` às numéricas. O One-Hot cria indicadores binários por categoria; `handle_unknown="ignore"` evita erro quando a validação contém categorias inéditas.
- `sparse_output=True` e `sparse_threshold=1.0` favorecem uma matriz esparsa, que armazena apenas valores não nulos quando aplicável. A representação pode alterar custo e comportamento numérico em relação ao baseline denso anterior.
- `RandomForestClassifier` constrói um conjunto de árvores com amostragem e seleção aleatória de features. `XGBClassifier` constrói árvores sequenciais para corrigir erros anteriores.
- `Pipeline` garante que o codificador e o escalonador sejam ajustados com `fit()` somente nos dados de treino; a validação passa apenas por `transform()`.
- Os `assert` finais impedem listas com features repetidas ou inexistentes.

**Conceito:** nesta comparação alteramos **somente o conjunto de features entre A, B e C**; os hiperparâmetros permanecem fixos por algoritmo. Isso permite atribuir diferenças principalmente à mudança de representação das variáveis, respeitando a variabilidade do treinamento.

### Etapa 4 — Resultados, interpretação e conclusão

#### Resultados obtidos

Foram definidos três cenários experimentais para investigar a contribuição incremental de variáveis geográficas e operacionais na previsão de atrasos de coleta.

| Cenário | Quantidade de features | Configuração |
|---|---:|---|
| A — Baseline | 10 | Variáveis utilizadas nos experimentos anteriores |
| B — Baseline + AOI | 11 | Baseline acrescido de `aoi_id` |
| C — Baseline + AOI + Courier | 12 | Cenário B acrescido de `courier_id` |

As dez variáveis originais foram preservadas em todos os cenários.

O planejamento contempla três configurações de features para dois algoritmos, totalizando **seis treinamentos**.

#### Interpretação

A organização incremental permite investigar se informações mais específicas sobre a localização e o entregador acrescentam capacidade preditiva ao modelo original.

A comparação entre os cenários A e B permitirá avaliar a contribuição de `aoi_id`, mantendo as demais features constantes.

A comparação entre B e C permitirá avaliar a contribuição adicional de `courier_id`, considerando que `aoi_id` já está presente.

Entretanto, esses experimentos não permitem determinar isoladamente o efeito de `courier_id` em relação ao baseline, nem avaliar explicitamente a interação entre entregador e região.

A quantidade de features representa as variáveis originais, não o número final de colunas após a codificação categórica.

#### Conclusão

**Etapa 4 concluída com sucesso.**

Os três cenários foram definidos conforme o planejamento e estão preparados para os experimentos comparativos.

Nenhum modelo foi treinado nesta etapa e o conjunto de teste permanece reservado.

A seleção das variáveis será fundamentada nos resultados de desempenho preditivo e custo computacional, sem presumir antecipadamente que o aumento da quantidade de features produzirá melhores resultados.

## 5 — Execução controlada dos seis experimentos

**Esta é a célula demorada.** Executa um modelo por vez, registra tempo, Average Precision e desempenho por capacidade operacional. Não faz tuning nem consulta o teste. Execute depois de conferir as saídas das células anteriores.

In [7]:
resultados = []
resultados_k = []
probs_validacao = {}
for nome_cenario, fs in cenarios.items():
    Xtr = treino[fs].copy()
    Xva = validacao[fs].copy()
    for algoritmo in ["Random Forest", "XGBoost"]:
        print(f"\nIniciando: {nome_cenario} | {algoritmo}", flush=True)
        pipeline = montar_pipeline(algoritmo, fs)
        inicio = time.perf_counter()
        pipeline.fit(Xtr, y_tr)
        segundos = time.perf_counter() - inicio
        probs = pipeline.predict_proba(Xva)[:, 1]
        assert len(probs) == len(y_va) and np.isfinite(probs).all() and ((probs >= 0) & (probs <= 1)).all()
        ap = average_precision_score(y_va, probs)
        chave = (nome_cenario, algoritmo)
        probs_validacao[chave] = probs.copy()
        resultados.append({"Cenário": nome_cenario, "Algoritmo": algoritmo, "Features": len(fs), "Average Precision": ap, "Tempo (s)": segundos})
        ordem = np.lexsort((np.arange(len(probs)), -probs))
        verdade = y_va.to_numpy()
        total_positivos = int(verdade.sum())
        for k in capacidades:
            n_alertas = int(np.ceil(len(probs) * k / 100))
            tp = int(verdade[ordem[:n_alertas]].sum())
            resultados_k.append({"Cenário": nome_cenario, "Algoritmo": algoritmo, "Capacidade (%)": k, "Alertas": n_alertas, "TP": tp, "Precision@K": tp / n_alertas, "Recall@K": tp / total_positivos})
        print(f"AP={ap:.6f} | Tempo={segundos:.1f}s", flush=True)
        del pipeline
        gc.collect()
    del Xtr, Xva
    gc.collect()
df_resultados = pd.DataFrame(resultados)
df_operacional = pd.DataFrame(resultados_k)
assert len(df_resultados) == 6 and len(df_operacional) == 42
print("\nTodos os seis experimentos concluídos.")


Iniciando: A — Baseline (10) | Random Forest
AP=0.179512 | Tempo=70.2s

Iniciando: A — Baseline (10) | XGBoost
AP=0.171407 | Tempo=6.7s

Iniciando: B — Baseline + AOI (11) | Random Forest
AP=0.149510 | Tempo=15.0s

Iniciando: B — Baseline + AOI (11) | XGBoost
AP=0.175768 | Tempo=7.0s

Iniciando: C — Baseline + AOI + Courier (12) | Random Forest
AP=0.209851 | Tempo=13.0s

Iniciando: C — Baseline + AOI + Courier (12) | XGBoost
AP=0.269095 | Tempo=7.8s

Todos os seis experimentos concluídos.


### Entendendo o Python — treinamento e avaliação operacional

1. O laço externo `for ... in cenarios.items()` percorre os três grupos de features; o interno percorre os dois algoritmos: **3 × 2 = 6 ajustes**.
2. `treino[fs].copy()` e `validacao[fs].copy()` selecionam exatamente as mesmas colunas em cada partição.
3. `pipeline.fit(Xtr, y_tr)` aprende transformações e modelo apenas no treino. `predict_proba(Xva)[:, 1]` retorna a probabilidade estimada da classe atraso na validação.
4. `np.isfinite()` e as verificações de intervalo confirmam probabilidades numéricas entre zero e um.
5. `average_precision_score(y_va, probs)` resume a curva precisão–recall ao longo de diferentes limiares. É útil para classes raras, mas não mede diretamente uma capacidade operacional fixa.
6. `np.lexsort((índice, -probs))` ordena por risco decrescente e usa o índice como desempate determinístico.
7. Para cada capacidade `k`, `np.ceil(N * k / 100)` calcula quantas coletas cabem na fila de alertas. `TP` conta atrasos reais entre os alertados; `Precision@K = TP / alertas` e `Recall@K = TP / total_de_atrasos`.
8. `time.perf_counter()` mede o tempo de `fit()`; não inclui o tempo de predição. Os resultados são acumulados em **listas de dicionários** e convertidos em DataFrames.
9. `del pipeline` e `gc.collect()` reduzem objetos retidos entre experimentos; `probs_validacao` guarda probabilidades dos seis cenários para eventual análise posterior, consumindo memória adicional.

**Atenção:** a célula executa os seis treinamentos em sequência. Como você quer validar uma célula por vez, confira primeiro a saída de configuração da etapa 4 e só então execute esta célula. Não há acesso ao teste.

### Etapa 5 — Resultados, interpretação e conclusão

#### Resultados obtidos

Foram executados seis experimentos de classificação, combinando três configurações de features com os algoritmos Random Forest e XGBoost.

| Cenário | Algoritmo | Average Precision | Tempo (s) |
|---|---|---:|---:|
| A — Baseline | Random Forest | 0,179512 | 78,3 |
| A — Baseline | XGBoost | 0,171407 | 7,2 |
| B — Baseline + AOI | Random Forest | 0,149510 | 16,8 |
| B — Baseline + AOI | XGBoost | 0,175768 | 7,4 |
| C — Baseline + AOI + Courier | Random Forest | 0,209851 | 14,2 |
| C — Baseline + AOI + Courier | XGBoost | **0,269095** | 8,0 |

O XGBoost no cenário C apresentou o maior valor de Average Precision.

Em comparação ao baseline, sua melhoria relativa foi de aproximadamente **57,0%**.

#### Interpretação

A inclusão isolada de `aoi_id` apresentou efeitos diferentes nos dois algoritmos.

Na Random Forest, a Average Precision diminuiu de 0,179512 para 0,149510. No XGBoost, aumentou de 0,171407 para 0,175768.

A inclusão adicional de `courier_id` produziu melhorias expressivas em ambos os algoritmos.

Esses resultados sugerem que a identificação do entregador contém informação preditiva relevante para o problema, quando considerada em conjunto com as demais features.

Entretanto, não é possível atribuir a melhoria exclusivamente à interação entre entregador e região, pois essa interação não foi modelada explicitamente.

Também não foi realizado um experimento utilizando `courier_id` sem `aoi_id`.

O XGBoost apresentou tempos de treinamento relativamente estáveis entre os cenários. Já a Random Forest apresentou diferenças expressivas de tempo, que exigem investigação antes de conclusões sobre eficiência computacional.

#### Conclusão

**Os seis experimentos foram concluídos com sucesso.**

O cenário C, com 12 features, apresentou o melhor desempenho em Average Precision para ambos os algoritmos.

O XGBoost foi o modelo com maior Average Precision entre os seis experimentos.

Os resultados reforçam a importância de investigar a contribuição das variáveis antes da otimização de hiperparâmetros.

A seleção definitiva das features ainda depende da análise das métricas operacionais, especialmente Precision@K e Recall@K, e de verificações adicionais de generalização.

O conjunto de teste permanece reservado para a avaliação final.

## 6 — Comparação estatística e operacional

Organizar as seis avaliações e medir o ganho de AP **em relação ao baseline reexecutado do mesmo algoritmo**, sem confundir ganho de feature com ganho de hiperparâmetros.

In [8]:
display(df_resultados.sort_values(["Algoritmo", "Average Precision"], ascending=[True, False]).reset_index(drop=True))
base_ap = df_resultados.loc[df_resultados["Cenário"] == "A — Baseline (10)"].set_index("Algoritmo")["Average Precision"]
comparacao = df_resultados.copy()
comparacao["AP baseline mesmo algoritmo"] = comparacao["Algoritmo"].map(base_ap)
comparacao["Delta AP"] = comparacao["Average Precision"] - comparacao["AP baseline mesmo algoritmo"]
display(comparacao.sort_values(["Algoritmo", "Cenário"]))
print("\nRecall@K por cenário e algoritmo:")
display(df_operacional.pivot_table(index="Capacidade (%)", columns=["Algoritmo", "Cenário"], values="Recall@K").round(4))
print("\nPrecision@K por cenário e algoritmo:")
display(df_operacional.pivot_table(index="Capacidade (%)", columns=["Algoritmo", "Cenário"], values="Precision@K").round(4))

,Cenário,Algoritmo,Features,Average Precision,Tempo (s)
0,C — Baseline + AOI + Courier (12),Random Forest,12,0.209851,12.963489
1,A — Baseline (10),Random Forest,10,0.179512,70.172319
2,B — Baseline + AOI (11),Random Forest,11,0.149510,15.029943
3,C — Baseline + AOI + Courier (12),XGBoost,12,0.269095,7.798294
4,B — Baseline + AOI (11),XGBoost,11,0.175768,7.029597
5,A — Baseline (10),XGBoost,10,0.171407,6.659678


,Cenário,Algoritmo,Features,Average Precision,Tempo (s),AP baseline mesmo algoritmo,Delta AP
0,A — Baseline (10),Random Forest,10,0.179512,70.172319,0.179512,0.000000
2,B — Baseline + AOI (11),Random Forest,11,0.149510,15.029943,0.179512,-0.030002
4,C — Baseline + AOI + Courier (12),Random Forest,12,0.209851,12.963489,0.179512,0.030340
1,A — Baseline (10),XGBoost,10,0.171407,6.659678,0.171407,0.000000
3,B — Baseline + AOI (11),XGBoost,11,0.175768,7.029597,0.171407,0.004361
5,C — Baseline + AOI + Courier (12),XGBoost,12,0.269095,7.798294,0.171407,0.097688



Recall@K por cenário e algoritmo:


Algoritmo          Random Forest                          \
Cenário        A — Baseline (10) B — Baseline + AOI (11)   
Capacidade (%)                                             
1                         0.1377                  0.1250   
2                         0.2036                  0.1802   
5                         0.3343                  0.3067   
10                        0.4593                  0.4150   
15                        0.5422                  0.5006   
20                        0.6054                  0.5652   
30                        0.7105                  0.6716   

Algoritmo                                                  XGBoost  \
Cenário        C — Baseline + AOI + Courier (12) A — Baseline (10)   
Capacidade (%)                                                       
1                                         0.1527            0.1377   
2                                         0.2529            0.2017   
5                                         0.4172            0.3205   
10                                        0.5389            0.4681   
15                                        0.6017            0.5514   
20                                        0.6397            0.6129   
30                                        0.7212            0.7157   

Algoritmo                                                                 
Cenário        B — Baseline + AOI (11) C — Baseline + AOI + Courier (12)  
Capacidade (%)                                                            
1                               0.1397                            0.1909  
2                               0.2004                            0.2917  
5                               0.3229                            0.4488  
10                              0.4620                            0.5562  
15                              0.5567                            0.6338  
20                              0.6171                            0.6806  
30                              0.7217                            0.7518


Precision@K por cenário e algoritmo:


Algoritmo          Random Forest                          \
Cenário        A — Baseline (10) B — Baseline + AOI (11)   
Capacidade (%)                                             
1                         0.3510                  0.3187   
2                         0.2595                  0.2298   
5                         0.1705                  0.1564   
10                        0.1171                  0.1058   
15                        0.0922                  0.0851   
20                        0.0772                  0.0721   
30                        0.0604                  0.0571   

Algoritmo                                                  XGBoost  \
Cenário        C — Baseline + AOI + Courier (12) A — Baseline (10)   
Capacidade (%)                                                       
1                                         0.3893            0.3510   
2                                         0.3225            0.2572   
5                                         0.2128            0.1635   
10                                        0.1374            0.1194   
15                                        0.1023            0.0937   
20                                        0.0816            0.0781   
30                                        0.0613            0.0608   

Algoritmo                                                                 
Cenário        B — Baseline + AOI (11) C — Baseline + AOI + Courier (12)  
Capacidade (%)                                                            
1                               0.3561                            0.4867  
2                               0.2556                            0.3719  
5                               0.1647                            0.2289  
10                              0.1178                            0.1418  
15                              0.0946                            0.1078  
20                              0.0787                            0.0868  
30                              0.0614                            0.0639

### Entendendo o Python — comparação justa

- `sort_values()` organiza os resultados sem alterar os modelos; `reset_index(drop=True)` renumera a tabela para exibição.
- `.loc[condição]` seleciona as linhas do cenário A. `set_index("Algoritmo")` permite localizar o AP de referência para cada algoritmo.
- `.map(base_ap)` atribui a cada experimento o AP do **baseline reexecutado com o mesmo algoritmo**.
- `Delta AP` é uma subtração em pontos absolutos: AP do cenário menos AP do baseline correspondente. Valor positivo indica ganho na validação; não prova, sozinho, ganho robusto fora da amostra.
- `pivot_table(index=..., columns=..., values=...)` transforma resultados longos em tabelas comparativas por capacidade, algoritmo e cenário.
- `round(4)` facilita a leitura, mas não altera os valores armazenados.

**Interpretação esperada:** observar AP global, Recall@K e Precision@K nas capacidades de 1%, 2%, 5%, 10%, 15%, 20% e 30%, além do tempo. Um cenário pode melhorar AP e piorar a captura de atrasos em uma capacidade operacional específica.

### Etapa 6 — Resultados, interpretação e conclusão

#### 6.1. Objetivo da análise

Consolidar os resultados dos seis experimentos para avaliar a contribuição incremental de `aoi_id` e `courier_id` na identificação antecipada de coletas com risco de atraso.

A comparação considera três dimensões:

- **Desempenho preditivo:** Average Precision.
- **Capacidade operacional:** Recall@K e Precision@K.
- **Custo computacional:** tempo de treinamento.

#### 6.2. Resultados da Average Precision

| Cenário | Random Forest | XGBoost |
|---|---:|---:|
| A — Baseline (10) | 0,179512 | 0,171407 |
| B — Baseline + AOI (11) | 0,149510 | 0,175768 |
| C — Baseline + AOI + Courier (12) | 0,209851 | **0,269095** |

O cenário C apresentou o melhor desempenho nos dois algoritmos.

O XGBoost obteve uma melhoria relativa de aproximadamente **57,0%** em Average Precision quando comparado ao baseline original.

A inclusão isolada de `aoi_id` não apresentou melhoria consistente entre os algoritmos.

Por outro lado, a inclusão adicional de `courier_id` produziu ganhos relevantes, sugerindo que a identificação do entregador contém informação preditiva complementar.

#### 6.3. Resultados operacionais

O cenário C apresentou os maiores valores de Recall@K e Precision@K em todas as capacidades avaliadas, para ambos os algoritmos.

No XGBoost, destacam-se:

| Capacidade | Recall baseline | Recall cenário C | Precision cenário C |
|---|---:|---:|---:|
| 1% | 13,77% | 19,09% | 48,67% |
| 5% | 32,05% | 44,88% | 22,89% |
| 10% | 46,81% | 55,62% | 14,18% |
| 20% | 61,29% | 68,06% | 8,68% |
| 30% | 71,57% | 75,18% | 6,39% |

Esses resultados indicam que o cenário C permite identificar uma parcela maior dos atrasos utilizando a mesma capacidade operacional de acompanhamento.

O ganho é especialmente relevante em capacidades menores, nas quais a Torre de Controle precisa selecionar um grupo reduzido de coletas prioritárias.

#### 6.4. Custo computacional

O XGBoost apresentou tempos de treinamento entre 7,17 e 8,03 segundos nos três cenários.

A Random Forest apresentou tempos entre 14,18 e 78,29 segundos.

As diferenças observadas na Random Forest não devem ser atribuídas exclusivamente à quantidade de features, pois podem envolver características das matrizes processadas e das condições de execução.

#### 6.5. Limitações

Os resultados foram obtidos no conjunto de validação e não representam ainda a avaliação final de generalização.

A inclusão de identificadores de alta cardinalidade, especialmente `courier_id`, pode favorecer o aprendizado de padrões associados a entregadores já observados.

Além disso, os experimentos não permitem isolar completamente o efeito de `courier_id`, pois sua inclusão foi avaliada sobre um cenário que já continha `aoi_id`.

Também não foi criada uma variável explícita de interação entre entregador e região.

#### 6.6. Conclusão

Os resultados demonstram que a seleção de features tem impacto relevante sobre o desempenho dos classificadores.

O **XGBoost com 12 features** apresentou o melhor desempenho estatístico e operacional entre os seis experimentos.

A inclusão de `aoi_id` e `courier_id` mostrou-se promissora para a previsão de atrasos de coleta.

O cenário C é, portanto, o principal candidato para os próximos experimentos.

Antes da otimização de hiperparâmetros, recomenda-se uma verificação complementar da contribuição individual de `courier_id` e da robustez dos resultados associados a categorias pouco frequentes.

O conjunto de teste permanece reservado para a avaliação final.

## 7 — Verificação de categorias inéditas na validação

Medir o percentual de registros de validação com AOI/entregador não observado no treinamento. Isso ajuda a interpretar limitações do One-Hot e do particionamento.

In [9]:
novidades = []
for coluna in ["region_id", "aoi_id", "courier_id"]:
    conhecidas = set(treino[coluna].dropna().unique())
    nao_vistas = ~validacao[coluna].isin(conhecidas)
    novidades.append({"Variável": coluna, "Categorias treino": len(conhecidas), "Categorias validação": validacao[coluna].nunique(), "Registros com categoria inédita": int(nao_vistas.sum()), "Percentual inédito (%)": 100 * nao_vistas.mean()})
display(pd.DataFrame(novidades))
print("Esta auditoria não altera a avaliação e não utiliza o teste.")

,Variável,Categorias treino,Categorias validação,Registros com categoria inédita,Percentual inédito (%)
0,region_id,30,30,0,0.000000
1,aoi_id,5664,4776,122,0.057108
2,courier_id,4232,3192,164,0.076769


Esta auditoria não altera a avaliação e não utiliza o teste.


### Entendendo o Python — categorias novas e generalização

- `set(treino[coluna].dropna().unique())` cria o conjunto de categorias vistas no treinamento, desconsiderando nulos.
- `validacao[coluna].isin(conhecidas)` retorna uma série booleana que indica se cada categoria já apareceu no treino; `~` inverte o resultado e marca as inéditas.
- `.sum()` sobre booleanos conta registros verdadeiros; `.mean()` calcula a fração de registros com categoria inédita; multiplicar por 100 transforma em porcentagem.
- `nunique()` compara quantas categorias distintas existem em cada partição, sem confundir isso com quantidade de registros.

**Limitação:** `handle_unknown="ignore"` permite prever mesmo quando uma categoria nunca apareceu no treino, mas o One-Hot não dispõe de uma coluna específica aprendida para ela. Isso não equivale a afirmar que o modelo não tem outras informações sobre a coleta: região, localização e horário continuam disponíveis. Uma taxa alta de categorias inéditas pode limitar o benefício de identificadores muito granulares.

### Etapa 7 — Resultados, interpretação e conclusão

#### 7.1. Resultados obtidos

Foi realizada uma auditoria para identificar registros do conjunto de validação cujas categorias de `region_id`, `aoi_id` e `courier_id` não estavam presentes no conjunto de treinamento.

| Variável | Categorias no treinamento | Categorias na validação | Registros com categoria inédita | Percentual inédito |
|---|---:|---:|---:|---:|
| `region_id` | 30 | 30 | 0 | 0,0000% |
| `aoi_id` | 5.664 | 4.776 | 122 | 0,0571% |
| `courier_id` | 4.232 | 3.192 | 164 | 0,0768% |

A auditoria foi realizada exclusivamente com os conjuntos de treinamento e validação.

Nenhum registro foi excluído ou modificado.

#### 7.2. Interpretação

A variável `region_id` apresentou cobertura completa das categorias observadas na validação.

As variáveis `aoi_id` e `courier_id` apresentaram percentuais muito baixos de registros com categorias inéditas.

Esses resultados indicam que a avaliação realizada representa predominantemente novas coletas associadas a regiões, AOIs e entregadores já observados durante o treinamento.

Esse cenário é compatível com a aplicação do modelo em uma operação logística estabelecida, na qual entregadores e áreas de coleta podem aparecer repetidamente.

Entretanto, a baixa incidência de categorias inéditas não elimina o risco de overfitting, especialmente para identificadores de alta cardinalidade.

Também não permite concluir que o modelo apresentará desempenho semelhante para entregadores ou AOIs nunca observados.

#### 7.3. Relação com os experimentos anteriores

Na etapa de comparação de features, o XGBoost com 12 variáveis apresentou Average Precision de 0,269095, superior ao baseline de 0,171407.

A auditoria atual mostra que esse ganho foi observado em uma validação composta predominantemente por categorias já conhecidas.

Portanto, os resultados sustentam o uso dessas variáveis como candidatas para previsão de novas coletas na operação conhecida, mas não demonstram generalização para categorias inéditas.

#### 7.4. Conclusão

**Etapa 7 concluída com sucesso.**

A auditoria identificou baixa incidência de categorias inéditas na validação:

- `region_id`: 0%.
- `aoi_id`: 0,0571%.
- `courier_id`: 0,0768%.

Os resultados não apresentam impedimento imediato para continuar os experimentos com as 12 features.

Entretanto, recomenda-se manter documentada a limitação de generalização para entregadores e AOIs não observados durante o treinamento.

O conjunto de teste permanece reservado para a avaliação final.

## 7.1 — Experimento complementar D: Baseline + Courier

### Objetivo

Avaliar a contribuição de `courier_id` isoladamente em relação às dez features originais, sem incluir `aoi_id`.

Esse experimento complementa os cenários A, B e C, formando uma comparação fatorial 2 × 2 entre a presença e a ausência de AOI e Courier.

| Cenário | AOI | Courier | Features |
|---|---|---|---:|
| A — Baseline | Não | Não | 10 |
| B — Baseline + AOI | Sim | Não | 11 |
| C — Baseline + AOI + Courier | Sim | Sim | 12 |
| D — Baseline + Courier | Não | Sim | 11 |

### Metodologia

Serão executados dois treinamentos adicionais:

1. Random Forest com as dez features originais mais `courier_id`.
2. XGBoost com as mesmas onze features.

Serão preservados os hiperparâmetros, o pré-processamento, os conjuntos de treinamento e validação e as métricas dos experimentos anteriores.

A avaliação utilizará Average Precision, Precision@K, Recall@K e tempo de treinamento, considerando capacidades operacionais de 1%, 2%, 5%, 10%, 15%, 20% e 30%.

**Hipótese investigada:** verificar se `courier_id` contribui para a identificação antecipada de coletas com risco de atraso independentemente da inclusão de `aoi_id`.

O experimento não pretende estabelecer causalidade nem demonstrar generalização para entregadores inéditos.

O conjunto de teste permanece reservado e não será utilizado.

In [10]:

# ============================================================
# ETAPA 7.1 — CENÁRIO D: BASELINE + COURIER
# ============================================================

nome_cenario = "D — Baseline + Courier (11)"
features_d = baseline + ["courier_id"]

# Verificar os resultados anteriores antes de acrescentar novos.
assert len(df_resultados) == 6, (
    "Esperados seis experimentos anteriores. "
    "Verifique o estado do notebook antes de continuar."
)
assert len(df_operacional) == 42
assert nome_cenario not in df_resultados["Cenário"].values
assert len(features_d) == 11
assert set(features_d).issubset(treino.columns)

# Preparar somente as features do cenário D.
Xtr_d = treino[features_d].copy()
Xva_d = validacao[features_d].copy()

# Guardar os resultados adicionais separadamente até a validação.
novos_resultados = []
novos_resultados_k = []
novas_probabilidades = {}

verdade = y_va.to_numpy()
total_positivos = int(verdade.sum())

for algoritmo in ["Random Forest", "XGBoost"]:

    print(f"\nIniciando: {nome_cenario} | {algoritmo}", flush=True)

    # Reutilizar o pipeline e os hiperparâmetros originais.
    pipeline = montar_pipeline(algoritmo, features_d)

    inicio = time.perf_counter()
    pipeline.fit(Xtr_d, y_tr)
    segundos = time.perf_counter() - inicio

    # Prever probabilidades somente na validação.
    probs = pipeline.predict_proba(Xva_d)[:, 1]

    assert len(probs) == len(y_va)
    assert np.isfinite(probs).all()
    assert ((probs >= 0) & (probs <= 1)).all()

    ap = average_precision_score(y_va, probs)

    chave = (nome_cenario, algoritmo)
    novas_probabilidades[chave] = probs.copy()

    novos_resultados.append({
        "Cenário": nome_cenario,
        "Algoritmo": algoritmo,
        "Features": len(features_d),
        "Average Precision": ap,
        "Tempo (s)": segundos
    })

    # Ordenar coletas por risco previsto, do maior para o menor.
    ordem = np.lexsort((np.arange(len(probs)), -probs))

    for k in capacidades:
        n_alertas = int(np.ceil(len(probs) * k / 100))
        tp = int(verdade[ordem[:n_alertas]].sum())

        novos_resultados_k.append({
            "Cenário": nome_cenario,
            "Algoritmo": algoritmo,
            "Capacidade (%)": k,
            "Alertas": n_alertas,
            "TP": tp,
            "Precision@K": tp / n_alertas,
            "Recall@K": tp / total_positivos
        })

    print(f"AP={ap:.6f} | Tempo={segundos:.1f}s", flush=True)

    del pipeline
    gc.collect()

# Validar os dois experimentos antes de consolidar.
assert len(novos_resultados) == 2
assert len(novos_resultados_k) == 14

# Atualizar as estruturas de comparação já existentes.
df_resultados = pd.concat(
    [df_resultados, pd.DataFrame(novos_resultados)],
    ignore_index=True
)

df_operacional = pd.concat(
    [df_operacional, pd.DataFrame(novos_resultados_k)],
    ignore_index=True
)

probs_validacao.update(novas_probabilidades)

assert len(df_resultados) == 8
assert len(df_operacional) == 56

print("\nCenário D concluído.")
print("Total: 8 experimentos e 56 avaliações operacionais.")

display(pd.DataFrame(novos_resultados))

print("\nRecall@K — Cenário D")
display(
    pd.DataFrame(novos_resultados_k)
    .pivot(
        index="Capacidade (%)",
        columns="Algoritmo",
        values="Recall@K"
    )
    .round(4)
)

print("\nPrecision@K — Cenário D")
display(
    pd.DataFrame(novos_resultados_k)
    .pivot(
        index="Capacidade (%)",
        columns="Algoritmo",
        values="Precision@K"
    )
    .round(4)
)

del Xtr_d, Xva_d
gc.collect()



Iniciando: D — Baseline + Courier (11) | Random Forest
AP=0.226512 | Tempo=17.6s

Iniciando: D — Baseline + Courier (11) | XGBoost
AP=0.268009 | Tempo=6.8s

Cenário D concluído.
Total: 8 experimentos e 56 avaliações operacionais.


,Cenário,Algoritmo,Features,Average Precision,Tempo (s)
0,D — Baseline + Courier (11),Random Forest,11,0.226512,17.624063
1,D — Baseline + Courier (11),XGBoost,11,0.268009,6.813450



Recall@K — Cenário D


Algoritmo,Random Forest,XGBoost
Capacidade (%),,
1,0.1634,0.1918
2,0.2709,0.2917
5,0.4348,0.4455
10,0.5365,0.5541
15,0.5967,0.6281
20,0.6441,0.6804
30,0.7313,0.7515



Precision@K — Cenário D


Algoritmo,Random Forest,XGBoost
Capacidade (%),,
1,0.4165,0.4890
2,0.3454,0.3719
5,0.2218,0.2272
10,0.1368,0.1413
15,0.1015,0.1068
20,0.0821,0.0868
30,0.0622,0.0639


0

### Entendendo o Python — Experimento complementar D

**1. Construção das features**

`features_d = baseline + ["courier_id"]` cria uma nova lista com as dez variáveis originais e o identificador do entregador.

A lista `baseline` permanece inalterada.

**2. Validação do estado do notebook**

As instruções `assert` verificam condições que precisam ser verdadeiras antes de iniciar os treinamentos.

Por exemplo, `len(df_resultados) == 6` confirma que os seis experimentos anteriores estão disponíveis e que o cenário D ainda não foi acrescentado.

Isso evita duplicações acidentais ou a consolidação de resultados incompletos.

**3. Reutilização do pipeline**

A função `montar_pipeline(algoritmo, features_d)` foi definida na Etapa 4.

Ela constrói o pré-processamento e o classificador correspondente, preservando os hiperparâmetros utilizados anteriormente.

A variável `courier_id` é tratada como categórica e transformada por One-Hot Encoding, com suporte a categorias não observadas no treinamento.

**4. Treinamento e previsão**

`pipeline.fit(Xtr_d, y_tr)` ajusta o modelo com os dados de treinamento.

`predict_proba(Xva_d)[:, 1]` retorna a probabilidade estimada para a classe positiva, que representa a coleta após o limite da janela operacional.

O conjunto de teste não participa desse processo.

**5. Average Precision**

`average_precision_score(y_va, probs)` calcula a Average Precision, que resume o desempenho da curva Precision-Recall.

Essa métrica é relevante porque os atrasos representam uma parcela pequena dos registros.

**6. Avaliação operacional por capacidade**

`np.lexsort()` organiza os registros por probabilidade prevista de atraso, em ordem decrescente, utilizando o índice como critério de desempate.

Para cada capacidade K, o código seleciona as coletas com maior risco previsto.

- **Precision@K:** proporção de atrasos reais entre as coletas selecionadas para alerta.
- **Recall@K:** proporção de todos os atrasos reais identificados entre as coletas selecionadas.

Isso permite analisar o desempenho em diferentes capacidades de atendimento da Torre de Controle.

**7. Preservação e consolidação dos resultados**

Os resultados adicionais são armazenados inicialmente em listas e dicionários separados.

Após as verificações, `pd.concat()` acrescenta as novas avaliações às tabelas anteriores, enquanto `update()` incorpora as probabilidades previstas.

Assim, os seis experimentos originais permanecem disponíveis para a comparação final.

**8. Controle de memória**

`del` remove referências a objetos temporários e `gc.collect()` solicita a coleta de objetos não utilizados.

Essas operações ajudam a controlar o consumo de memória, especialmente durante treinamentos sucessivos.

### Resultado esperado

Ao final da execução, deverão existir:

- 8 experimentos registrados.
- 56 avaliações operacionais: 8 modelos × 7 capacidades.
- Métricas de Average Precision e tempo para os dois modelos do cenário D.
- Tabelas de Recall@K e Precision@K para o cenário D.

A interpretação e a conclusão serão documentadas após a execução, utilizando os resultados efetivamente observados.

### 7.1 — Resultados, interpretação e conclusão

#### Resultados obtidos

O cenário D, composto pelas dez features originais e `courier_id`, foi executado com sucesso nos algoritmos Random Forest e XGBoost.

| Algoritmo | Features | Average Precision | Tempo de treinamento |
|---|---:|---:|---:|
| Random Forest | 11 | 0,226512 | 17,62 s |
| XGBoost | 11 | 0,268009 | 6,81 s |

Foram concluídos os oito experimentos planejados, totalizando 56 avaliações operacionais.

#### Comparação com os cenários anteriores

**Random Forest**

- A — Baseline: AP = 0,179512.
- B — Baseline + AOI: AP = 0,149510.
- C — Baseline + AOI + Courier: AP = 0,209851.
- D — Baseline + Courier: AP = 0,226512.

O cenário D apresentou a maior Average Precision entre as quatro configurações do Random Forest.

Em relação ao baseline, o ganho absoluto foi de 0,047000, equivalente a aproximadamente 26,2% de melhoria relativa.

A inclusão de AOI ao cenário com Courier não produziu ganho adicional de AP neste experimento.

**XGBoost**

- A — Baseline: AP = 0,171407.
- B — Baseline + AOI: AP = 0,175768.
- C — Baseline + AOI + Courier: AP = 0,269095.
- D — Baseline + Courier: AP = 0,268009.

O cenário C apresentou a maior Average Precision, mas a diferença em relação ao cenário D foi de apenas 0,001086.

O cenário D superou o baseline em 0,096602 de AP, correspondendo a aproximadamente 56,4% de melhoria relativa.

#### Avaliação operacional

No cenário D, o Random Forest apresentou Recall@K de:

- 1%: 0,1634.
- 2%: 0,2709.
- 5%: 0,4348.
- 10%: 0,5365.
- 15%: 0,5967.
- 20%: 0,6441.
- 30%: 0,7313.

O XGBoost apresentou Recall@K de:

- 1%: 0,1918.
- 2%: 0,2917.
- 5%: 0,4455.
- 10%: 0,5541.
- 15%: 0,6281.
- 20%: 0,6804.
- 30%: 0,7515.

Em termos operacionais, o XGBoost D apresentou maior Recall@K que o Random Forest D em todas as capacidades analisadas, exceto em nenhuma: na capacidade de 2%, os valores foram 0,2917 e 0,2709, respectivamente, mantendo a vantagem do XGBoost.

Comparando os cenários C e D, o XGBoost C apresentou pequenas vantagens nas capacidades de 5% a 30%, enquanto o D apresentou vantagem em 1% e empate nos valores arredondados de 2%.

No Random Forest, o cenário D apresentou melhor Recall@K em 1%, 2%, 5%, 20% e 30%, enquanto o cenário C foi superior em 10% e 15%.

As diferenças operacionais devem ser avaliadas considerando as prioridades e a capacidade de atuação da Torre de Controle.

#### Interpretação

Os resultados demonstram que a inclusão de `courier_id` está associada a ganhos relevantes de capacidade preditiva nos dois algoritmos.

Entretanto, a contribuição adicional de `aoi_id` depende do algoritmo e da métrica considerada.

No Random Forest, a combinação de AOI e Courier apresentou desempenho inferior ao uso isolado de Courier em Average Precision.

No XGBoost, a combinação apresentou a maior Average Precision, mas com diferença pequena em relação ao uso isolado de Courier.

Esses resultados não demonstram causalidade, nem permitem concluir que o identificador do entregador representa diretamente sua eficiência individual.

Também é necessário considerar a alta cardinalidade de `courier_id`, o risco de aprendizado de padrões específicos e a baixa representatividade de entregadores inéditos na validação.

#### Conclusão

**O cenário D foi concluído e validado.**

A comparação fatorial dos quatro cenários revelou que:

1. A inclusão isolada de Courier melhora significativamente a Average Precision em relação ao baseline nos dois algoritmos.
2. A inclusão isolada de AOI apresenta resultados distintos entre Random Forest e XGBoost.
3. A combinação AOI + Courier não garante melhor desempenho em todos os algoritmos ou capacidades operacionais.
4. Random Forest D e XGBoost C apresentaram as maiores Average Precision de seus respectivos algoritmos.
5. XGBoost D permanece um candidato competitivo, devido à proximidade de desempenho em relação ao XGBoost C e à utilização de uma feature a menos.

A escolha definitiva das configurações que seguirão para otimização será realizada na Etapa 8, considerando Average Precision, Recall@K, Precision@K, custo computacional e limitações metodológicas.

O conjunto de teste permanece reservado para a avaliação final.

## 8 — Consolidação final dos experimentos e decisão sobre as features

### 8.1. Objetivo

Consolidar os experimentos de Random Forest e XGBoost para avaliar a contribuição de `aoi_id` e `courier_id` na identificação antecipada de coletas com risco de ocorrer após o limite da janela operacional.

Foram avaliados quatro cenários de features em dois algoritmos, totalizando oito experimentos.

A comparação utiliza os mesmos conjuntos de treinamento e validação, os mesmos hiperparâmetros por algoritmo e o mesmo critério principal de avaliação: **Average Precision (AP)**.

O conjunto de teste permanece reservado.

### 8.2. Cenários avaliados

| Cenário | Configuração | Features |
|---|---|---:|
| A | Baseline | 10 |
| B | Baseline + `aoi_id` | 11 |
| C | Baseline + `aoi_id` + `courier_id` | 12 |
| D | Baseline + `courier_id` | 11 |

A comparação constitui um experimento fatorial 2 × 2 de inclusão de identificadores.

### 8.3. Resultados consolidados — Average Precision

| Cenário | Random Forest | XGBoost |
|---|---:|---:|
| A | 0,179512 | 0,171407 |
| B | 0,149510 | 0,175768 |
| C | 0,209851 | **0,269095** |
| D | **0,226512** | 0,268009 |

**Random Forest**

O cenário D apresentou a maior AP do algoritmo, com ganho absoluto de 0,047000 em relação ao baseline, equivalente a aproximadamente 26,2% de melhoria relativa.

A inclusão isolada de AOI reduziu a AP. A combinação AOI + Courier melhorou o baseline, mas ficou abaixo da configuração com somente Courier.

**XGBoost**

O cenário C apresentou a maior AP geral, com ganho absoluto de 0,097688 sobre o baseline, aproximadamente 57,0% de melhoria relativa.

Entretanto, o cenário D alcançou AP de 0,268009, apenas 0,001086 abaixo do cenário C.

Portanto, a inclusão adicional de AOI trouxe ganho pequeno de AP quando Courier já estava presente.

### 8.4. Avaliação operacional — Recall@K

| Capacidade | RF C | RF D | XGB C | XGB D |
|---|---:|---:|---:|---:|
| 1% | 0,1527 | **0,1634** | 0,1909 | **0,1918** |
| 2% | 0,2529 | **0,2709** | 0,2917 | 0,2917 |
| 5% | 0,4172 | **0,4348** | **0,4488** | 0,4455 |
| 10% | **0,5389** | 0,5365 | **0,5562** | 0,5541 |
| 15% | **0,6017** | 0,5967 | **0,6338** | 0,6281 |
| 20% | 0,6397 | **0,6441** | **0,6806** | 0,6804 |
| 30% | 0,7212 | **0,7313** | **0,7518** | 0,7515 |

Os cenários C e D superaram amplamente os cenários A e B nas capacidades operacionais analisadas.

O XGBoost C apresentou vantagem sobre D nas capacidades de 5% a 30%, enquanto D foi ligeiramente superior em 1%. Na capacidade de 2%, os valores arredondados foram iguais.

No Random Forest, D foi superior em cinco das sete capacidades, enquanto C foi superior em 10% e 15%.

Essas diferenças demonstram a importância de considerar a capacidade operacional da Torre de Controle, e não somente uma métrica global.

### 8.5. Avaliação operacional — Precision@K

| Capacidade | RF C | RF D | XGB C | XGB D |
|---|---:|---:|---:|---:|
| 1% | 0,3893 | **0,4165** | 0,4867 | **0,4890** |
| 2% | 0,3225 | **0,3454** | 0,3719 | 0,3719 |
| 5% | 0,2128 | **0,2218** | **0,2289** | 0,2272 |
| 10% | **0,1374** | 0,1368 | **0,1418** | 0,1413 |
| 15% | **0,1023** | 0,1015 | **0,1078** | 0,1068 |
| 20% | 0,0816 | **0,0821** | 0,0868 | 0,0868 |
| 30% | 0,0613 | **0,0622** | 0,0639 | 0,0639 |

A Precision@K confirma que os cenários C e D apresentam desempenho operacional próximo, especialmente no XGBoost.

Na capacidade de 10%, por exemplo, o XGBoost C apresentou Precision@K de 0,1418 e Recall@K de 0,5562.

Isso significa que, ao priorizar aproximadamente 10% das coletas com maior risco previsto, o modelo identificou aproximadamente 55,62% dos atrasos reais presentes na validação.

A precisão de 14,18% deve ser interpretada considerando a baixa prevalência de atrasos na população, aproximadamente 2,55%.

### 8.6. Cardinalidade e categorias inéditas

A auditoria identificou:

| Variável | Categorias no treino | Registros inéditos na validação | Percentual inédito |
|---|---:|---:|---:|
| `region_id` | 30 | 0 | 0,0000% |
| `aoi_id` | 5.664 | 122 | 0,0571% |
| `courier_id` | 4.232 | 164 | 0,0768% |

A maioria dos registros de validação pertence a categorias já observadas no treinamento.

Esse resultado é compatível com uma aplicação em uma operação logística conhecida, mas não comprova desempenho equivalente para novas AOIs ou novos entregadores.

As categorias raras também exigem cautela, pois padrões aprendidos a partir de poucas observações podem apresentar menor estabilidade.

### 8.7. Custo computacional e complexidade

O XGBoost apresentou tempos de treinamento menores que o Random Forest nos experimentos observados.

No cenário D, os tempos foram aproximadamente:

- Random Forest: 17,62 segundos.
- XGBoost: 6,81 segundos.

Esses tempos representam medições pontuais, não benchmarks repetidos. Diferenças entre execuções podem decorrer de condições do ambiente, representação dos dados e outros fatores computacionais.

A configuração D utiliza uma feature original a menos que C e elimina a necessidade de codificar `aoi_id`, variável de alta cardinalidade.

Entretanto, o menor número de features originais não permite quantificar, isoladamente, a redução de memória ou do número de colunas após One-Hot Encoding.

### 8.8. Limitações metodológicas

Os resultados devem ser interpretados considerando que:

1. Os modelos foram comparados em um único particionamento estratificado por linha.
2. A validação foi utilizada para selecionar as configurações candidatas, podendo produzir algum otimismo na escolha.
3. Não foi realizada validação específica com grupos inteiramente inéditos de entregadores ou AOIs.
4. Identificadores de alta cardinalidade podem favorecer o aprendizado de padrões particulares.
5. Não foram realizados experimentos repetidos suficientes para estimar a estabilidade estatística das pequenas diferenças entre cenários.
6. A comparação histórica com a etapa 2.24 deve considerar as diferenças anteriormente identificadas entre representações esparsas e densas.
7. A inclusão de identificadores não demonstra causalidade nem importância relativa das regiões.
8. As features utilizadas deverão continuar disponíveis no momento real da previsão, evitando data leakage.

### 8.9. Decisão técnica

Com base nos resultados, ficam selecionadas as seguintes configurações candidatas:

**Random Forest — Cenário D**

Baseline + `courier_id`, com 11 features.

Justificativa: maior Average Precision do Random Forest e desempenho competitivo nas capacidades operacionais analisadas.

**XGBoost — Cenário C**

Baseline + `aoi_id` + `courier_id`, com 12 features.

Justificativa: maior Average Precision geral e melhores resultados em diversas capacidades operacionais relevantes.

**XGBoost — Cenário D**

Baseline + `courier_id`, com 11 features.

Justificativa: desempenho muito próximo ao cenário C, menor complexidade de entrada e pequena vantagem operacional na capacidade de 1%.

Os demais cenários permanecem documentados como histórico experimental e referência de comparação.

A seleção representa uma decisão para a próxima fase de investigação, não uma confirmação definitiva de superioridade.

### 8.10. Conclusão final

A investigação demonstrou que `courier_id` apresenta contribuição preditiva relevante no conjunto de validação utilizado.

A variável `aoi_id`, por outro lado, apresentou contribuição adicional dependente do algoritmo e da configuração de features.

O XGBoost C alcançou a maior Average Precision geral, enquanto o XGBoost D apresentou desempenho muito próximo com menor quantidade de features originais.

No Random Forest, o cenário D apresentou a melhor Average Precision.

**A etapa de investigação e comparação de features está concluída.**

### 8.11. Próxima etapa — Otimização de hiperparâmetros

A próxima fase será o planejamento do `RandomizedSearchCV`, incluindo:

- Definição dos modelos e cenários candidatos.
- Escolha dos hiperparâmetros e intervalos de busca.
- Definição do número de configurações e folds.
- Controle de memória e tempo computacional.
- Seleção por Average Precision, com análise complementar de métricas operacionais.
- Preservação do conjunto de teste para a avaliação final.

Nenhuma otimização será iniciada antes da aprovação do planejamento.

**O conjunto de teste permanece intacto.**